# 07 - Identification diagnostics

This notebook checks whether the buffered staggered DiD design has enough support and credible comparability before estimation.

In [14]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.estimation_workflow_utils import (
    append_tag_to_filename,
    cohort_year_support,
    standardized_mean_differences,
)

DATA_DIR = PROJECT_ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for path in [TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)


Project root: c:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2


## User configuration

In [15]:
RUN_TAG = '1km'
START_YEAR = 2001
TREATMENT_KEY = 'protected_area'
CELL_COL = 'cell_id'
YEAR_COL = 'year'
OUTCOME_COL = 'loss_m2'

BUFFER_RADII_KM = [10, 25, 50]
MAIN_BUFFER_KM = 25
PRETREND_EVENT_WINDOW = range(-8, 0)

PANEL_PATH = INTERMEDIATE_DIR / f'panel_treatment_{RUN_TAG}.parquet'
if not PANEL_PATH.exists():
    PANEL_PATH = INTERMEDIATE_DIR / 'panel_treatment.parquet'
EXPOSURE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('06_nearest_treated_exposure.parquet', RUN_TAG)
USE_CORE_COVARIATES = False
USE_ROBUST_COVARIATES = False
CORE_COVARIATE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('panel_core_covariates.parquet', RUN_TAG)
ROBUST_COVARIATE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('panel_robustness_covariates.parquet', RUN_TAG)

SUPPORT_TABLE_NAME = '07_cohort_support_after_buffering.csv'
PRETREND_TABLE_NAME = '07_pretrend_summary.csv'
BALANCE_TABLE_NAME = '07_covariate_balance_by_buffer.csv'
OVERLAP_TABLE_NAME = '07_propensity_overlap_summary.csv'
SUPPORT_FIG_NAME = '07_buffered_control_support.png'
PRETREND_FIG_NAME = '07_pretrend_profiles.png'


## Load inputs

In [ ]:
import pyarrow.parquet as pq

first_treat_col = 'first_treat_year'
panel_schema_cols = pq.ParquetFile(PANEL_PATH).schema_arrow.names
treated_col = f'treated_it_{TREATMENT_KEY}'
if treated_col not in panel_schema_cols:
    treated_col = 'treated_it'

panel_cols = [CELL_COL, YEAR_COL, OUTCOME_COL, first_treat_col, treated_col, 'treated_before_panel_start', 'never_treated', 'ever_treated', 'event_time']
panel = pd.read_parquet(PANEL_PATH, columns=panel_cols)
panel[CELL_COL] = panel[CELL_COL].astype('string')
panel[YEAR_COL] = pd.to_numeric(panel[YEAR_COL], errors='coerce').astype(int)
panel = panel[panel[YEAR_COL] >= START_YEAR].copy()

exposure = pd.read_parquet(EXPOSURE_PATH)
exposure[CELL_COL] = exposure[CELL_COL].astype('string')
exposure[YEAR_COL] = pd.to_numeric(exposure[YEAR_COL], errors='coerce').astype(int)

core_covs = pd.read_parquet(CORE_COVARIATE_PATH) if USE_CORE_COVARIATES and CORE_COVARIATE_PATH.exists() else pd.DataFrame()
robust_covs = pd.read_parquet(ROBUST_COVARIATE_PATH) if USE_ROBUST_COVARIATES and ROBUST_COVARIATE_PATH.exists() else pd.DataFrame()
print('Panel rows:', f'{len(panel):,}')
print('Exposure rows:', f'{len(exposure):,}')
print('Core covariates:', core_covs.shape, '| enabled:', USE_CORE_COVARIATES, '| path:', CORE_COVARIATE_PATH)
print('Robust covariates:', robust_covs.shape, '| enabled:', USE_ROBUST_COVARIATES, '| path:', ROBUST_COVARIATE_PATH)


## A. Cohort support after buffering

In [ ]:
support_tables = []
for buffer_km in BUFFER_RADII_KM:
    support_tables.append(
        cohort_year_support(
            panel,
            exposure,
            buffer_km=buffer_km,
            cell_col=CELL_COL,
            year_col=YEAR_COL,
            first_treat_col=first_treat_col,
            treated_col=treated_col,
            treated_before_panel_col='treated_before_panel_start',
            min_year=START_YEAR,
        )
    )
support = pd.concat(support_tables, ignore_index=True)
support_path = TABLE_DIR / append_tag_to_filename(SUPPORT_TABLE_NAME, RUN_TAG)
support.to_csv(support_path, index=False)

support_summary = support.groupby('buffer_km').agg(
    min_controls=('n_eligible_control_cells', 'min'),
    p10_controls=('n_eligible_control_cells', lambda x: x.quantile(0.10)),
    median_controls=('n_eligible_control_cells', 'median'),
    unsupported_cells=('n_eligible_control_cells', lambda x: int((x == 0).sum())),
).reset_index()
print('Saved:', support_path)
print(support_summary.to_string(index=False))


## B. Pre-trend profiles

In [ ]:
buffer_cols = [f'outside_{int(buffer_km)}km_buffer' for buffer_km in BUFFER_RADII_KM]
main_exp = exposure[[CELL_COL, YEAR_COL] + buffer_cols].copy()
df = panel.merge(main_exp, on=[CELL_COL, YEAR_COL], how='left')
df['first_treat_year_num'] = pd.to_numeric(df[first_treat_col], errors='coerce')
df = df[~df['treated_before_panel_start'].fillna(False).astype(bool)].copy()

pretrend_rows = []
cohorts = sorted(g for g in df['first_treat_year_num'].dropna().unique() if g > START_YEAR)
years_available = set(df[YEAR_COL].unique())
for buffer_km in BUFFER_RADII_KM:
    outside_col = f'outside_{int(buffer_km)}km_buffer'
    for g in cohorts:
        treated_ids = df.loc[df['first_treat_year_num'] == g, CELL_COL].drop_duplicates()
        for event_time in PRETREND_EVENT_WINDOW:
            year = int(g + event_time)
            if year not in years_available:
                continue
            year_df = df[df[YEAR_COL] == year]
            treated = year_df[year_df[CELL_COL].isin(treated_ids)]
            controls = year_df[
                ((year_df['first_treat_year_num'].isna()) | (year_df['first_treat_year_num'] > year))
                & (pd.to_numeric(year_df[treated_col], errors='coerce').fillna(0).astype(int) == 0)
                & year_df[outside_col].fillna(False)
            ]
            pretrend_rows.append({
                'buffer_km': float(buffer_km),
                'cohort_year': int(g),
                'event_time': int(event_time),
                'year': year,
                'treated_mean_loss_m2': treated[OUTCOME_COL].mean(),
                'control_mean_loss_m2': controls[OUTCOME_COL].mean(),
                'difference_m2': treated[OUTCOME_COL].mean() - controls[OUTCOME_COL].mean(),
                'n_treated_rows': int(len(treated)),
                'n_control_rows': int(len(controls)),
            })
pretrend = pd.DataFrame(pretrend_rows)
pretrend_summary = pretrend.groupby(['buffer_km', 'event_time']).agg(
    treated_mean_loss_m2=('treated_mean_loss_m2', 'mean'),
    control_mean_loss_m2=('control_mean_loss_m2', 'mean'),
    difference_m2=('difference_m2', 'mean'),
    n_cohorts=('cohort_year', 'nunique'),
    median_control_rows=('n_control_rows', 'median'),
).reset_index()
pretrend_path = TABLE_DIR / append_tag_to_filename(PRETREND_TABLE_NAME, RUN_TAG)
pretrend.to_csv(pretrend_path, index=False)
print('Saved:', pretrend_path)
print(pretrend_summary[pretrend_summary['buffer_km'] == float(MAIN_BUFFER_KM)].to_string(index=False))


## C. Covariate balance and optional propensity overlap

In [ ]:
balance_rows = []
overlap_rows = []
cov_sets = {}
if USE_CORE_COVARIATES and not core_covs.empty:
    cov_sets['core'] = core_covs
if USE_ROBUST_COVARIATES and not robust_covs.empty:
    cov_sets['robust'] = robust_covs

for cov_set_name, cov_df in cov_sets.items():
    if cov_df.empty:
        continue
    cov_df = cov_df.copy()
    cov_df[CELL_COL] = cov_df[CELL_COL].astype('string')
    covariates = [c for c in cov_df.columns if c not in {CELL_COL, 'cell_lon', 'cell_lat', 'first_treat_year', 'ever_treated', 'never_treated'}]
    for buffer_km in BUFFER_RADII_KM:
        clean_control_ids = exposure.loc[exposure[f'outside_{int(buffer_km)}km_buffer'], CELL_COL].drop_duplicates()
        treatment_status = panel[[CELL_COL, 'ever_treated']].drop_duplicates(subset=[CELL_COL]).copy()
        bal = cov_df.drop(columns=['ever_treated', 'never_treated'], errors='ignore').merge(treatment_status, on=CELL_COL, how='left')
        bal['ever_treated_for_balance'] = bal['ever_treated'].fillna(False).astype(bool)
        bal = bal[bal['ever_treated_for_balance'] | (bal[CELL_COL].isin(clean_control_ids))].copy()
        smd = standardized_mean_differences(bal, group_col='ever_treated_for_balance', covariates=covariates, treated_value=True)
        smd['buffer_km'] = buffer_km
        smd['covariate_set'] = cov_set_name
        balance_rows.append(smd)

        try:
            from sklearn.linear_model import LogisticRegression
            from sklearn.pipeline import make_pipeline
            from sklearn.preprocessing import StandardScaler
            model_df = bal[[CELL_COL, 'ever_treated_for_balance'] + covariates].dropna().copy()
            if model_df['ever_treated_for_balance'].nunique() == 2 and len(model_df) > 100:
                X = model_df[covariates].to_numpy(dtype=float)
                y = model_df['ever_treated_for_balance'].astype(int).to_numpy()
                model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, n_jobs=-1))
                model.fit(X, y)
                ps = model.predict_proba(X)[:, 1]
                overlap_rows.append({
                    'covariate_set': cov_set_name,
                    'buffer_km': buffer_km,
                    'n_model_rows': len(model_df),
                    'pscore_p01': float(np.quantile(ps, 0.01)),
                    'pscore_p50': float(np.quantile(ps, 0.50)),
                    'pscore_p99': float(np.quantile(ps, 0.99)),
                    'share_pscore_lt_0p05': float((ps < 0.05).mean()),
                    'share_pscore_gt_0p95': float((ps > 0.95).mean()),
                })
        except Exception as exc:
            overlap_rows.append({'covariate_set': cov_set_name, 'buffer_km': buffer_km, 'error': repr(exc)})

balance = pd.concat(balance_rows, ignore_index=True) if balance_rows else pd.DataFrame()
overlap = pd.DataFrame(overlap_rows)
balance_path = TABLE_DIR / append_tag_to_filename(BALANCE_TABLE_NAME, RUN_TAG)
overlap_path = TABLE_DIR / append_tag_to_filename(OVERLAP_TABLE_NAME, RUN_TAG)
balance.to_csv(balance_path, index=False)
overlap.to_csv(overlap_path, index=False)
if not cov_sets:
    print('Covariate balance skipped because USE_CORE_COVARIATES and USE_ROBUST_COVARIATES are both False, or no tag-matched covariate files were found.')
print('Saved:', balance_path)
print('Saved:', overlap_path)
print(balance.head(12).to_string(index=False) if not balance.empty else 'No covariate balance table created.')
print(overlap.to_string(index=False) if not overlap.empty else 'No overlap table created.')


## Diagnostic figures

In [ ]:
fig_path = FIG_DIR / append_tag_to_filename(SUPPORT_FIG_NAME, RUN_TAG)
ax = support_summary.plot(x='buffer_km', y=['min_controls', 'p10_controls', 'median_controls'], marker='o', figsize=(8, 5))
ax.set_title('Eligible controls after spatial buffering')
ax.set_xlabel('Buffer radius (km)')
ax.set_ylabel('Control cells per cohort-year')
ax.grid(axis='y', alpha=0.3)
ax.xaxis.grid(False)
ax.legend(frameon=False)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(fig_path, dpi=220, bbox_inches='tight')
plt.show()

fig_path = FIG_DIR / append_tag_to_filename(PRETREND_FIG_NAME, RUN_TAG)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
main_pretrend = pretrend_summary[pretrend_summary['buffer_km'] == float(MAIN_BUFFER_KM)].copy()
main_pretrend.plot(x='event_time', y=['treated_mean_loss_m2', 'control_mean_loss_m2'], marker='o', ax=axes[0])
axes[0].axvline(-1, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title(f'Treated vs controls, {MAIN_BUFFER_KM:g}km buffer')
axes[0].set_xlabel('Event time')
axes[0].set_ylabel('Mean annual forest loss (m2)')
axes[0].grid(axis='y', alpha=0.3)
axes[0].xaxis.grid(False)
axes[0].legend(frameon=False)

difference_plot = pretrend_summary.pivot(index='event_time', columns='buffer_km', values='difference_m2')
difference_plot.plot(marker='o', ax=axes[1])
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axvline(-1, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Treated-control pretrend gap by buffer')
axes[1].set_xlabel('Event time')
axes[1].set_ylabel('Difference in mean annual forest loss (m2)')
axes[1].grid(axis='y', alpha=0.3)
axes[1].xaxis.grid(False)
axes[1].legend(title='Buffer km', frameon=False)
fig.tight_layout()
fig.savefig(fig_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved diagnostic figures.')
